In [1]:
from typing import List
import numpy as np
import scipy.sparse as sp

from Components.Controllers import LTI_LQRc_RHG

/Users/leonardchanfreau/Documents/ETHZ/Master Thesis/code/GroundWater_RGH/Components/Controllers/LTI_LQRc_RHG.py:9: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
# Reload scripts when executed
%load_ext autoreload
%autoreload 2

In [3]:
A = np.array([[1, 1], [0, 1]])  # (2x2)

B0 = np.array([[1], [0]])
B1 = np.array([[0], [1]])
B2 = np.array([[1, 0], [1, 1]])
B_stack = [B0, B1, B2]
B_minus_v = [np.hstack([B1,B2]), np.hstack([B0,B2]), np.hstack([B0,B1])]

C = np.array([[1], [0]])

In [4]:
nx = 2
nu = 4
Nv = 3
nu_v = [1, 1, 2]
K=3

In [5]:
def form_B_tilde(A_powers: np.ndarray, B: np.ndarray) -> List[sp.lil_array]:
    B_tilde_v = []
    # Loop over agents
    for v, B_v in enumerate(B):
        # Compute all needed A^k B: shape (K, nx, nu)
        AkB = np.einsum('kij,jv->kiv', A_powers, B_v)
        # Lower triangular indices
        i_idx, j_idx = np.tril_indices(K)

        # Fetch the right A^k B for each block
        blocks = AkB[i_idx - j_idx]  # shape (num_blocks, nx, nu)

        # Step 4: Place blocks into final array
        result = sp.lil_array(((K+1) * nx, K * nu_v[v]))
        for idx, (i, j) in enumerate(zip(i_idx, j_idx)):
            result[(i+1)*nx:(i+2)*nx, j*nu_v[v]:(j+1)*nu_v[v]] = blocks[idx]
        B_tilde_v.append(result)
    
    return B_tilde_v

In [6]:
A_powers = np.array([np.linalg.matrix_power(A, k) for k in range(K + 1)])
A_powers

array([[[1, 0],
        [0, 1]],

       [[1, 1],
        [0, 1]],

       [[1, 2],
        [0, 1]],

       [[1, 3],
        [0, 1]]])

In [7]:
B_tilde = form_B_tilde(A_powers, B_stack)
[mat.toarray() for mat in B_tilde]

[array([[0., 0., 0.],
        [0., 0., 0.],
        [1., 0., 0.],
        [0., 0., 0.],
        [1., 1., 0.],
        [0., 0., 0.],
        [1., 1., 1.],
        [0., 0., 0.]]),
 array([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [1., 1., 0.],
        [2., 1., 0.],
        [1., 1., 1.]]),
 array([[0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [2., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [3., 2., 2., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])]

In [33]:
Q_v = [np.array([[1,0], [0,i]]) for i in range(3)]
R_v = [np.array([[i]]) for i in range(2)] + [np.array([[2, 0], [0, 1]])]
M_v = [np.array([[i], [1]]) for i in range(2)] + [np.array([[2, 0], [0, 1]])]
L_v = [np.array([[1]]) for _ in range(2)] + [np.array([[0, 1]])]

/Users/leonardchanfreau/Documents/ETHZ/Master Thesis/code/GroundWater_RGH/Components/Controllers/LTI_LQRc_RHG.py:9: SyntaxWarning: invalid escape sequence '\ '
  """
/Users/leonardchanfreau/Documents/ETHZ/Master Thesis/code/GroundWater_RGH/Components/Controllers/LTI_LQRc_RHG.py:9: SyntaxWarning: invalid escape sequence '\ '
  """


In [9]:
H_x = np.zeros((2,2))
h_x = np.ones((2,1))
H_u = np.zeros((4,4))
h_u = np.ones((4,1))

In [52]:
ctrl = LTI_LQRc_RHG(K, nx, nu, Nv, nu_v,
                 A, B_stack, B_minus_v, C,
                 Q_v, R_v, M_v, L_v,
                 H_x, h_x, H_u, h_u)
x_0 = np.array([[0], [0]])
ctrl.step(x_0)

array([[-0.01559142],
       [ 1.16646745],
       [-1.26484815],
       [ 0.15122318],
       [-0.22705667],
       [-0.40946269],
       [-0.01884679],
       [-0.6958751 ],
       [ 0.20001806],
       [-0.68403625],
       [ 0.32376137],
       [-0.55947528]])